# Unrecoverable Mutations: DataFrames

This notebook demonstrates the distinction between recoverable column-level writes and unrecoverable mutations that bypass FlowBook's column tracking.

In [13]:
import pandas as pd
df = pd.DataFrame({'price': [10, 20], 'qty': [1, 2]})

In [14]:
total = df['price'].sum()  # Reads df['price']

In [15]:
df['discount'] = df['price'] * 0.1  # Column write — RECOVERABLE (tracked by FlowBook)

Adding or replacing a column via `df['col'] = ...` is a **recoverable** operation. FlowBook tracks column-level reads and writes, so it knows exactly which columns were modified and can determine staleness at column granularity. Re-running this cell will always produce the same `discount` column.

In [16]:
df.values[0, 0] = 999  # Bypasses column tracking — UNRECOVERABLE

Accessing `df.values` returns the underlying NumPy array and mutates it directly. This bypasses pandas' column API entirely, so FlowBook cannot detect which column was affected. The mutation to `price[0]` (now 999) is invisible to column-level tracking and cannot be recovered by re-running any single cell.

In [ ]:
df.reset_index(inplace=True)

❌ Wrote `df.index` read by @D

In [ ]:
df.index = ['a', 'b']  # Index mutation — UNRECOVERABLE

❌ Wrote `df.price` read by @B

Mutating the DataFrame's index is another form of structural change that FlowBook's column-level tracking does not cover. If this cell is deleted, the index change persists and cannot be undone by re-running other cells.

## Summary

| Operation | Example | Recoverable? |
|---|---|---|
| Column assignment | `df['col'] = ...` | Yes — tracked at column level |
| Direct array access | `df.values[i, j] = ...` | No — bypasses column tracking |
| Index mutation | `df.index = ...` | No — structural change, not column-level |